# Lab 05: The Multiagent Pattern for Evidence and Custody Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danjethh/sectratek-AI-Lab/blob/main/lab05_multiagent_pattern/03_lab_notebook.ipynb)

**Course:** AI for Cybersecurity and Digital Forensics  
**Instructor:** Daniel Kwaku Ntiamoah Addai, Mobile Threat Research Lead, SectraTEK Lab  
**Runs in:** Google Colab or any hosted Jupyter. Nothing is installed on your own computer.

---

In the first four labs, **one** AI agent worked each case on its own. (An **agent** is an AI model
that can use tools, such as "open this evidence file", before it answers.)

Real casework is often checked by more than one person. One examiner builds the timeline, a
second examiner checks it against the records, and a third looks at how the evidence was handled.
This lab copies that idea with a **team of three AI agents**. This is called the **Multiagent
Pattern**: each agent has one narrow job, and what it writes is passed on to the next agent.

The three team members:

- **InvestigationAgent** builds the timeline of what happened on the phone.
- **EvidenceVerificationAgent** checks those claims against the records.
- **CustodyAuditAgent** checks how the evidence was handled after it was collected, then writes
  the final, carefully worded conclusion.

How this notebook is laid out:

- **Part 0:** setup. Connect to the AI, write the case files, load the team code.
- **The Case in One Minute** and **How a Multiagent Team Works:** short reading, no code.
- **Part A:** you run the three agents yourself, one at a time, and watch what each one passes on.
- **Part B:** a **Crew** (a team manager) runs the same team for you, in the right order.
- **Your Final Judgment:** you write your own conclusion. The agents assist; you, the examiner,
  decide.

Run the cells from top to bottom. Each step uses what the step before it produced.

## Part 0: One-Time Setup

Run these four steps in order. Together they:

1. install one small helper library (Step 0.1),
2. connect the notebook to a free AI model that runs online at a company called Groq (Step 0.2),
3. write the case evidence files into a folder (Step 0.3),
4. load the code that lets AI agents work as a team (Step 0.4).

Everything happens inside this online session. Nothing is installed on your own computer.

### Step 0.1: Install the client library

**Purpose:** install the one extra library this notebook needs, so it can send messages to the AI
model.

**Input:** nothing from you. This is the first step.

**How it works:** `%pip install` downloads the `openai` library into this online session only.
Groq's service understands the same message format as OpenAI's, so this one library works with
Groq too.

**Output:** a short install log. Warnings are normal. Step 0.2 uses this library to connect to
the AI model.

In [ ]:
# Install the one helper library this notebook needs (only inside this online session).
# openai: a library for sending messages to an AI model. Groq accepts the same format.
%pip install -q openai

### Step 0.2: Connect to a free hosted model (Groq)

**Purpose:** connect the notebook to a free AI model. All three agents in this lab use it to read
the evidence and write their answers.

**Input:** the `openai` library from Step 0.1, plus a free **API key** (a private password that
lets the notebook use the AI model). To get one:

1. Open **https://console.groq.com/keys**, sign in, and create a key.
2. Run the cell and paste the key when asked. It stays hidden as you type and is **not saved**
   into the notebook.

**How it works:** the cell builds a **client** (the notebook's connection to Groq), picks the first
AI model on its list that your key can use, and sends a one-word test message. It also adds a small
safety wrapper: it keeps each reply short enough for the free plan, waits and tries again if Groq
says you are sending too fast, and asks again if the model replies in the wrong form.

**Output:** a line like `Model OK: qwen/... -> ready`. The connection is saved as `client` and the
model name as `MODEL`. Step A.2 gives both of them to every agent.

In [ ]:
# Connect to a free AI model hosted by Groq, then send a test message to prove it works.
import os        # os: lets the notebook read and store settings, such as your key
import getpass   # getpass: lets you type your secret key without it showing on screen
from openai import OpenAI  # OpenAI: sends your messages to the AI model (Groq accepts the same format)

# Ask for your key only if it is not already stored in this session.
if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass('Paste your Groq API key (hidden): ').strip()

# The web address of Groq's service.
BASE_URL = 'https://api.groq.com/openai/v1'
# A list of free AI models to try, best first. Groq retires models from time to time, so this
# cell uses the first one your key can reach and prints its name. Edit the list to try another.
# Qwen is first for this lab: our agents write their steps as plain-text tags such as <tool_call>,
# and Qwen follows that format well. gpt-oss sometimes plans silently and sends back an empty reply.
PREFERRED = ['qwen/qwen3.8-27b', 'openai/gpt-oss-120b']

# Build the client: the notebook's connection to Groq.
# max_retries=8: the free plan allows only so much text per minute (about 8,000 "tokens"; a token
# is a small piece of a word). If we go over, Groq says "try again in a few seconds", and the
# client waits and retries instead of stopping with an error.
client = OpenAI(base_url=BASE_URL, api_key=os.environ['GROQ_API_KEY'], max_retries=8)

# A small safety wrapper. Every message this notebook sends to the AI goes through it.
# It does two jobs:
# 1. It caps how long each reply can be: 1,500 tokens, or 900 for Qwen, whose free plan allows
#    only 1,000 reply tokens a minute. Groq counts this cap against the per-minute budget.
#    (A single call can still choose its own cap.)
# 2. gpt-oss sometimes tries to use a built-in feature instead of writing text, and Groq rejects
#    that with the error "model called a tool". The wrapper then asks again (up to 3 tries in
#    all) with a reminder to reply in plain text.
_create = client.chat.completions.create
def _create_within_limits(*args, **kw):
    kw.setdefault('max_tokens', 900 if 'qwen' in str(kw.get('model')) else 1500)
    for attempt in range(3):
        try:
            return _create(*args, **kw)
        except Exception as err:
            if 'called a tool' not in str(err) or attempt == 2:
                raise
            kw['messages'] = list(kw.get('messages', [])) + [
                {'role': 'user', 'content': 'Reply in plain text only; do not call functions.'}]
client.chat.completions.create = _create_within_limits   # from now on, every call uses the wrapper

# Ask Groq which models your key can use, and pick the first one from our list.
available = [m.id for m in client.models.list().data]
MODEL = next((m for m in PREFERRED if m in available), None)
# If none of our preferred models is available, print the ones that are, and stop here.
if MODEL is None:
    print('None of PREFERRED is on your key right now. Models your key can use:')
    for m in available:
        print('  -', m)
    raise SystemExit('Put one of these chat models first in PREFERRED and re-run this cell.')

# Send a one-word test message to prove the connection works.
probe = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': 'Reply with one word: ready'}])
print('Model OK:', MODEL, '->', (probe.choices[0].message.content or '').strip())

### Step 0.3: Write the synthetic case files

**Purpose:** put the case evidence on disk, so the agents have real records to read. This is case
`MA-2026-022`.

**Input:** the six evidence files, written out in full inside the code cell below. They are
**synthetic** (made up for teaching) and identical to the copies in the course repository's
`data/` folder.

| File | What it records |
|---|---|
| `artifact_manifest.json` | A case summary: the device, how it was collected, the incident window. |
| `device_state.csv` | When the phone was locked and unlocked. |
| `file_events.csv` | When the file `patients_contacts.png` was created. |
| `messaging_events.csv` | The SecureChat app's attempt to attach that file. |
| `network_events.csv` | The SecureChat upload record. |
| `chain_of_custody.csv` | Who handled the evidence after collection, and when. |

**How it works:** the cell keeps each file's text in a dictionary called `ARTIFACTS` (a list of
file name -> file text pairs). It makes a folder called `data/` and writes each file into it. A
file that is already there is left as it is.

**Output:** one line per file showing where it is saved, such as `wrote data/device_state.csv`.
The evidence tools in Step A.1 read these files.

In [ ]:
# Write the six case evidence files into a folder called data/, so the agents have records to read.
from pathlib import Path  # Path: a simple way to point at files and folders by name

# ARTIFACTS is a dictionary (a list of name -> value pairs): file name -> the file's full text.
ARTIFACTS = {'artifact_manifest.json': '{\n  "case_id": "MA-2026-022",\n  "lab": "Lab 05 - Multiagent Pattern",\n  "device": {\n    "platform": "Android",\n    "model": "Motorola Edge 2024",\n    "os_version": "Android 14"\n  },\n  "acquisition": {\n    "type": "full_file_system",\n    "tool": "Cellebrite UFED 7.70",\n    "acquired_at_utc": "2026-03-03T12:04:55Z",\n    "original_image_sha256": "2ef6f6d371459ce23d3953196b03f979f8d2d440d4e8f1bf13d1712d7bf7f3a1"\n  },\n  "analysis_timezone": "America/New_York",\n  "incident_window_utc": {\n    "start": "2026-03-01T18:35:00Z",\n    "end": "2026-03-01T19:10:00Z"\n  },\n  "note": "Synthetic classroom dataset for multiagent-pattern demo"\n}\n', 'device_state.csv': 'timestamp_utc,event,notes\n2026-03-01T18:35:02Z,device_locked,"Phone left unattended during vaccination event"\n2026-03-01T19:10:08Z,device_unlocked,"Phone recovered by staff member"\n', 'file_events.csv': 'timestamp_utc,path,event,sha256\n2026-03-01T18:44:11Z,/storage/emulated/0/Pictures/patients_contacts.png,created,91c01aa8ea5bd91ab2d4f0a2c2f9f8189517588fb9f76ff4e777e7345b4870ac\n', 'messaging_events.csv': 'timestamp_utc,app,event,target,details\n2026-03-01T18:45:02Z,SecureChat,attach_attempt,external_contact_7,"patients_contacts.png selected as outbound attachment"\n', 'network_events.csv': 'timestamp_utc,app,event,bytes_out,notes\n2026-03-01T18:45:07Z,SecureChat,upload_started,48211,"Outbound upload session opened"\n', 'chain_of_custody.csv': 'timestamp_utc,handler,action,notes\n2026-03-03T12:04:55Z,Investigator_A,acquired_image,"UFED acquisition to encrypted evidence drive"\n2026-03-03T12:41:08Z,Investigator_A,computed_hash,"SHA-256 verified against acquisition log"\n2026-03-04T09:15:30Z,Analyst_B,created_working_copy,"Exported device state, file, messaging, and network artifacts"\n2026-03-05T11:02:44Z,Analyst_C,review_started,"Review began, but no transfer entry from Analyst_B was documented"\n'}

# Make the data/ folder (no error if it already exists).
data_dir = Path('data'); data_dir.mkdir(exist_ok=True)
# A loop: repeat the indented lines once for each file.
for name, text in ARTIFACTS.items():
    fp = data_dir / name
    # Only write the file if it is not there yet, so an existing copy is never overwritten.
    if not fp.exists():
        fp.write_text(text)
    print(('wrote ' if fp.stat().st_size else 'exists ') + str(fp))

### Step 0.4: Load the multiagent framework

**Purpose:** load the code that turns the AI model into agents that can use tools and work as a
team.

**Input:** nothing new. This cell only defines building blocks. Later steps give them the `client`
and `MODEL` from Step 0.2.

**How it works:** the cell defines four building blocks:

- `@tool`: a label you put on an ordinary Python function to turn it into a **tool** the agent can
  ask to run.
- `ReactAgent`: one AI worker that repeats a simple loop: think, use one tool, read the result,
  until it can answer. (This is the ReAct pattern from Lab 03.)
- `Agent`: one **team member**. It wraps a ReactAgent and adds a role, a task, the headings its
  answer must use, and the teammates whose answers it receives.
- `Crew`: a **team manager** that runs the agents in the right order (used in Part B).

The cell is long. You do not need to read every line; skim the names. It is a self-contained copy
adapted from The Neural Maze's agentic-patterns-course (MIT License).

**Output:** nothing is printed. The building blocks are now ready for Part A.

In [ ]:
# The multiagent toolbox: @tool, ReactAgent, Agent, and Crew. Self-contained copy for this notebook.
# Adapted from The Neural Maze agentic-patterns-course (MIT License), (c) 2024 The Neural Maze.
#
# This is the toolbox. Skim the section names; you do not need to follow every line.
# Nothing runs yet: this cell only defines the building blocks that Part A and Part B use.

import json
import re
from dataclasses import dataclass
from typing import Callable


# ---------------------------------------------------------------- 1. Reading the AI's tags
# The AI writes each step inside tags, like <thought>...</thought> or <tool_call>...</tool_call>.
# The code below finds the text inside a tag, so the notebook knows what the AI decided to do.

# A small container: the text found inside the tags, and whether any tag was found at all.
@dataclass
class TagContentResult:
    content: list
    found: bool


def extract_tag_content(text: str, tag: str) -> TagContentResult:
    """Find the text inside <tag>...</tag> in the AI's reply (for <thought>, <tool_call>, <response>)."""
    # Tidy-up: some models repeat the closing </tool_call> tag several times; keep just one.
    if tag == "tool_call":
        text = re.sub(r"(</tool_call>\s*){2,}$", "</tool_call>", text.strip())
    pattern = rf"<{tag}>(.*?)</{tag}>"
    matches = re.findall(pattern, text, re.DOTALL)
    return TagContentResult(content=[m.strip() for m in matches], found=bool(matches))


# ---------------------------------------------------------------- 2. The conversation record
# The AI has no memory of its own. Each time, we send it the whole conversation so far: a list of
# messages, each marked with who wrote it ("system" = standing instructions, "user" = us,
# "assistant" = the AI).

# Package one message as {'role': who is speaking, 'content': the text}, optionally inside a tag.
def build_prompt_structure(prompt: str, role: str, tag: str = "") -> dict:
    if tag:
        prompt = f"<{tag}>{prompt}</{tag}>"
    return {"role": role, "content": prompt}


# Add one more message to the end of the conversation.
def update_chat_history(history: list, msg: str, role: str):
    history.append(build_prompt_structure(prompt=msg, role=role))


# The conversation itself: a list that can drop its oldest message when full (no limit by default).
class ChatHistory(list):
    def __init__(self, messages=None, total_length=-1):
        super().__init__(messages or [])
        self.total_length = total_length

    def append(self, msg):
        if len(self) == self.total_length:
            self.pop(0)
        super().append(msg)


# Send the whole conversation to the AI model and return its reply as plain text.
def completions_create(client, messages, model) -> str:
    response = client.chat.completions.create(messages=list(messages), model=model)
    return str(response.choices[0].message.content)


# ---------------------------------------------------------------- 3. The tool wrapper (@tool)
# These pieces turn an ordinary Python function into a tool the agent can ask to run.

# Describe a function for the AI: its name, its description (the docstring), and its inputs.
def get_fn_signature(fn: Callable) -> dict:
    sig = {"name": fn.__name__, "description": fn.__doc__, "parameters": {"properties": {}}}
    schema = {k: {"type": v.__name__} for k, v in fn.__annotations__.items() if k != "return"}
    sig["parameters"]["properties"] = schema
    return sig


# Before a tool runs, convert the inputs the AI asked for into the right type (for example "3" -> 3).
# Inputs the tool does not have are ignored.
def validate_arguments(tool_call: dict, tool_signature: dict) -> dict:
    properties = tool_signature["parameters"]["properties"]
    type_mapping = {"int": int, "str": str, "bool": bool, "float": float}
    validated = {}
    for name, value in tool_call["arguments"].items():
        schema = properties.get(name)
        if schema is None:
            continue
        converter = type_mapping.get(schema.get("type"))
        if converter is None:
            validated[name] = value
            continue
        if not isinstance(value, converter):
            value = converter(value)
        validated[name] = value
    tool_call["arguments"] = validated
    return tool_call


# A tool: its name, the function to run, and the description the AI reads.
class Tool:
    def __init__(self, name, fn, fn_signature):
        self.name = name
        self.fn = fn
        self.fn_signature = fn_signature

    def __str__(self):
        return self.fn_signature

    def run(self, **kwargs):
        return self.fn(**kwargs)


def tool(fn: Callable) -> Tool:
    """The @tool label: turn a plain function into a Tool, using its name, docstring, and inputs."""
    sig = get_fn_signature(fn)
    return Tool(name=sig["name"], fn=fn, fn_signature=json.dumps(sig))


# ---------------------------------------------------------------- 4. One AI worker (ReactAgent)
# The standing instructions every agent receives (its "system prompt"). They teach the AI the
# loop (think, use one tool, read the result) and the tag format to write it in. This text is
# sent to the AI exactly as written.
REACT_SYSTEM_PROMPT = """
## Role
You are a function-calling AI model that reasons and acts in a loop until you can deliver a final response to the user.

## Loop Structure
Each iteration of your loop consists of three steps:

1. **Thought** - Reason about the current situation and decide what to do next. Wrap in <thought></thought> tags.
2. **Action** - Call one tool by emitting a <tool_call> JSON object (see format below).
3. **Observation** - The tool result will be returned wrapped in <observation></observation> tags. Use it to inform your next thought.

Repeat this loop until you have enough information to give a final answer, then wrap it in <response></response> tags.

## Tool Call Format
<tool_call>
{"name": <function-name>, "arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

- Do NOT make assumptions about parameter values.
- Pay close attention to each parameter's `type` and use the correct Python types.
- Use exactly one tool call per turn, and increment `id` for each call.

## Available Tools
<tools>
%s
</tools>

## Example Session
<question>What's the current temperature in Madrid?</question>
<thought>I need to retrieve the current weather in Madrid.</thought>
<tool_call>{"name": "get_current_weather", "arguments": {"location": "Madrid", "unit": "celsius"}, "id": 0}</tool_call>
<observation>{0: {"temperature": 25, "unit": "celsius"}}</observation>
<thought>I now have the temperature for Madrid. I can respond to the user.</thought>
<response>The current temperature in Madrid is 25 degrees Celsius.</response>

## Additional Constraints
- Do not skip directly to a final answer while important evidence is still missing.
- Keep the final answer evidence-bounded: state only what the observations support.
- Never leave the loop without eventually producing a <response>.
"""


class ReactAgent:
    """One AI worker. It loops: think, call one tool, read the result (the "observation"),
    and repeats until it writes its final answer inside <response> tags."""

    # Set up the worker: its connection to the AI, the model name, its instructions, its tools.
    def __init__(self, tools, model="openai/gpt-oss-120b", client=None, system_prompt=""):
        if client is None:
            raise ValueError("ReactAgent requires a `client` (chat.completions.create).")
        self.client = client
        self.model = model
        self.system_prompt = system_prompt
        self.tools = tools if isinstance(tools, list) else [tools]
        self.tools_dict = {t.name: t for t in self.tools}

    # Join the descriptions of all this worker's tools, so they can go into its instructions.
    def add_tool_signatures(self) -> str:
        return "".join(t.fn_signature for t in self.tools)

    # Run every tool the AI asked for and collect the results ("observations").
    # An unknown tool name or a wrong input is reported back to the AI as text, so it can correct
    # itself; a garbled request is skipped. Either way, the notebook does not crash.
    def process_tool_calls(self, tool_calls_content: list) -> dict:
        observations = {}
        for tool_call_str in tool_calls_content:
            try:
                tool_call = json.loads(tool_call_str)
            except json.JSONDecodeError:
                print(f"\nSkipping malformed tool call: {tool_call_str}")
                continue
            tool_name = tool_call.get("name") if isinstance(tool_call, dict) else None
            t = self.tools_dict.get(tool_name)
            if t is None:
                print(f"\nSkipping unknown tool request: {tool_name}")
                observations[len(observations)] = f"Unknown tool {tool_name!r}. Use one of: {', '.join(self.tools_dict)}."
                continue
            print(f"\nUsing Tool: {tool_name}")
            tool_call.setdefault("arguments", {})
            try:
                validated = validate_arguments(tool_call, json.loads(t.fn_signature))
                print(f"Tool call: {validated}")
                result = t.run(**validated["arguments"])
            except Exception as e:   # a wrong or missing argument becomes an observation, not a crash
                validated, result = tool_call, f"Tool error: {e}"
            print(f"Tool result: {result}")
            observations[validated.get("id", len(observations))] = result
        return observations

    # The main loop. It goes round at most max_rounds times (10 by default). If there is still no
    # final answer after that, it asks the AI once more and returns that reply.
    def run(self, user_msg: str, max_rounds: int = 10) -> str:
        # Start the conversation: the standing instructions, then the task (the "question").
        system_prompt = (self.system_prompt + "\n" + REACT_SYSTEM_PROMPT % self.add_tool_signatures()).strip()
        chat_history = ChatHistory([
            build_prompt_structure(system_prompt, "system"),
            build_prompt_structure(user_msg, "user", tag="question"),
        ])
        tools_used, pushbacks = 0, 0
        for _ in range(max_rounds):
            # Send the conversation so far, and get the AI's next reply.
            completion = completions_create(self.client, chat_history, self.model)

            # Pull out any thought, tool request, or final answer from that reply.

            thought = extract_tag_content(str(completion), "thought")
            tool_calls = extract_tag_content(str(completion), "tool_call")
            response = extract_tag_content(str(completion), "response")
            if not str(completion).strip():       # an empty reply: remind the AI to write its tags
                update_chat_history(chat_history, "Your reply was empty. Write your <thought> and "
                                    "<tool_call> tags as plain text in the reply.", "user")
                continue
            # A final answer only counts when no tool request came with it. A model that asks for a
            # tool, imagines the result, and answers all at once must run the tool and see the REAL
            # result first.
            if response.found and not tool_calls.found:
                if self.tools_dict and tools_used == 0 and pushbacks < 2:
                    # Evidence check: an answer given before any tool has run cannot be based on the
                    # evidence, so send it back (at most twice) and ask the AI to use a tool first.
                    pushbacks += 1
                    update_chat_history(chat_history, completion, "assistant")
                    update_chat_history(chat_history, "You have not called any tool yet, so that answer is "
                                        "not based on evidence. Call a tool first with a <tool_call> block.", "user")
                    continue
                return response.content[0]
            # If the AI asked for a tool, cut off anything it wrote after the request (such as an
            # imagined result), then add its reply to the conversation.
            if tool_calls.found:
                completion = str(completion).split("<observation>")[0].split("<response>")[0]
            update_chat_history(chat_history, completion, "assistant")

            # Print the AI's thinking, so you can follow along.
            if thought.found and thought.content:
                print(f"\nThought: {thought.content[0]}")

            # Run the requested tools and send the real results back to the AI as an observation.
            if tool_calls.found:
                observations = self.process_tool_calls(tool_calls.content)
                tools_used += len(observations)
                print(f"\nObservations: {observations}")
                update_chat_history(chat_history, f"<observation>{observations}</observation>", "user")

        # Out of rounds: ask the AI one last time and return whatever it says.
        return completions_create(self.client, chat_history, self.model)


# ---------------------------------------------------------------- 5. The team: Agent and Crew
from textwrap import dedent


class Agent:
    """One team member: a ReactAgent plus a role (backstory), a task, and the teammates it depends on.

    Teammates are connected with the `>>` sign: a >> b means b depends on a, and a's answer is
    passed to b as context (background notes b reads before it starts).
    Adapted from The Neural Maze's multiagent_pattern/agent.py.
    """

    def __init__(self, name, backstory, task_description, task_expected_output="",
                 tools=None, llm="openai/gpt-oss-120b", client=None):
        self.name = name
        self.backstory = backstory
        self.task_description = task_description
        self.task_expected_output = task_expected_output
        self.react_agent = ReactAgent(model=llm, tools=tools or [], client=client)
        self.react_agent.system_prompt = backstory  # the role goes first in the worker's instructions
        self.dependencies = []
        self.dependents = []
        self.context = ""
        Crew.register_agent(self)   # if a Crew is open (Part B), join it

    def __repr__(self):
        return self.name

    # This is what makes "a >> b" work: it records that b depends on a.
    def __rshift__(self, other):
        self.add_dependent(other)
        return other

    # Record the connection in both directions: who I depend on, and who depends on me.
    def add_dependency(self, other):
        if isinstance(other, Agent):
            self.dependencies.append(other); other.dependents.append(self)
        elif isinstance(other, list):
            for item in other:
                self.dependencies.append(item); item.dependents.append(self)
        else:
            raise TypeError("dependency must be an Agent or list of Agents")

    def add_dependent(self, other):
        if isinstance(other, Agent):
            other.dependencies.append(self); self.dependents.append(other)
        elif isinstance(other, list):
            for item in other:
                item.dependencies.append(self); self.dependents.append(item)
        else:
            raise TypeError("dependent must be an Agent or list of Agents")

    # Store an earlier teammate's answer, to be included in this agent's task later.
    def receive_context(self, input_data):
        self.context += f"\n{self.name} received context:\n{input_data}\n"

    # Build the full task message for this agent: its task, the answer headings, and the context.
    # Everything inside the triple quotes below is text that is sent to the AI.
    def create_prompt(self):
        return dedent(f"""
        You are an AI agent on a team working together to complete a forensic task.
        The task is in <task_description></task_description>. Context from earlier agents is in
        <context></context> (empty if none). The required output format is in
        <task_expected_output></task_expected_output>. Produce the best response, following that format.

        <task_description>
        {self.task_description}
        </task_description>

        <task_expected_output>
        {self.task_expected_output}
        </task_expected_output>

        <context>
        {self.context}
        </context>

        Your response:
        """).strip()

    # Do the task, then pass the answer on to every teammate that depends on this agent.
    def run(self):
        output = self.react_agent.run(user_msg=self.create_prompt())
        for dependent in self.dependents:
            dependent.receive_context(output)
        return output


class Crew:
    """The team manager: holds a team of agents and runs each one only after its inputs are ready.

    Adapted from The Neural Maze's multiagent_pattern/crew.py (the picture-drawing part is left
    out; use `describe()` for a text view of who depends on whom).
    """

    current_crew = None

    def __init__(self):
        self.agents = []

    # "with Crew() as crew:" opens the crew; agents created inside the block join it.
    def __enter__(self):
        Crew.current_crew = self
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        Crew.current_crew = None

    def add_agent(self, agent):
        self.agents.append(agent)

    @staticmethod
    def register_agent(agent):
        if Crew.current_crew is not None:
            Crew.current_crew.add_agent(agent)

    # Work out a safe run order: an agent runs only after every agent it depends on has finished.
    # (The name for this kind of ordering is a "topological sort".) A loop of agents that all wait
    # on each other can never start, so that is reported as an error.
    def topological_sort(self):
        from collections import deque
        in_degree = {a: len(a.dependencies) for a in self.agents}
        queue = deque([a for a in self.agents if in_degree[a] == 0])
        ordered = []
        while queue:
            a = queue.popleft()
            ordered.append(a)
            for dep in a.dependents:
                in_degree[dep] -= 1
                if in_degree[dep] == 0:
                    queue.append(dep)
        if len(ordered) != len(self.agents):
            raise ValueError("Circular dependency detected among agents")
        return ordered

    # A plain-text summary: who depends on whom, and the run order.
    def describe(self):
        lines = ["Dependency graph (who feeds whom):"]
        for a in self.agents:
            deps = ", ".join(d.name for d in a.dependencies) or "(none)"
            lines.append(f"  {a.name}  <- depends on: {deps}")
        lines.append("Run order: " + " -> ".join(a.name for a in self.topological_sort()))
        return "\n".join(lines)

    # Run every agent in the safe order and collect their answers, looked up by agent name.
    def run(self):
        outputs = {}
        for agent in self.topological_sort():
            print(f"\n{'=' * 50}\nRUNNING AGENT: {agent.name}\n{'=' * 50}")
            out = agent.run()
            outputs[agent.name] = out
            print(out)
        return outputs

## The Case in One Minute

**Case `MA-2026-022`:** a public-health outreach phone was left unattended during a vaccination
event. While nobody was watching it, a file of patient contacts appeared and the phone tried to
send it out. The questions:

1. Was `patients_contacts.png` sent out (transmitted) from the phone?
2. How sure can we be? The answer must use one label: **confirmed**, **likely**, or **unconfirmed**.
3. Does a gap in the **chain-of-custody** record (the log of who held the evidence, and when)
   weaken that confidence?

| UTC time | What the records show |
|---|---|
| 18:35:02 | device locked (start of the unattended interval) |
| 18:44:11 | `patients_contacts.png` created on the device |
| 18:45:02 | SecureChat attach attempt with that file |
| 18:45:07 | network `upload_started` (48,211 bytes out) |
| 19:10:08 | device unlocked |
| (later) | an analyst review begins with **no documented transfer** from the prior handler |

Two points to keep in mind all the way through:

- The network record says the upload **started** (`upload_started`). No record says it
  **finished**. So the most the evidence supports is **likely**, not **confirmed**.
- The custody gap is about how the evidence was **handled after it was collected**. It is an
  evidence-handling issue, not proof about what the phone did.

Keeping those two questions separate is the heart of this lab.

## How a Multiagent Team Works

Three specialists, each with a narrow job. Each one hands its written answer forward:

| Agent | Its job | What it hands forward |
|---|---|---|
| **InvestigationAgent** | Rebuild the technical timeline from the phone's records. | A draft transmission assessment. |
| **EvidenceVerificationAgent** | Test those claims against the records. | A revised, evidence-bounded rating. |
| **CustodyAuditAgent** | Review how the evidence was handled; write the conclusion. | The final answer, with its confidence stated. |

Who feeds whom:

```text
InvestigationAgent ---> EvidenceVerificationAgent ---> CustodyAuditAgent
        |                                                     ^
        +-----------------------------------------------------+
```

Each agent works the way the agent in Lab 03 did: it thinks, uses a tool to open a record, reads
the result, and repeats until it can answer. What is new is the **handoff**. When an agent
finishes, its answer is passed automatically to the agents that depend on it, as **context**
(background notes the next agent reads before it starts). The later agents are told to test and
narrow the earlier claims, not just agree with them.

## Part A: Run the Team One Agent at a Time

In Part A you build the team and run each agent yourself, so you can see what each one writes and
what it passes on:

- **Step A.1** gives the agents their evidence tools.
- **Step A.2** creates the three agents and connects them.
- **Steps A.3 to A.5** run the agents one by one, in order.

### Step A.1: Define the evidence tools

**Purpose:** give the agents a simple, safe way to read the evidence. Each **tool** is a small
function that opens one evidence file and returns its rows.

**Input:** the six files in `data/` from Step 0.3, and the `@tool` label from Step 0.4.

**How it works:** the cell defines six tools. One lists the evidence files, four read the phone's
records (lock and unlock, the file event, the attach attempt, the upload record), and one reads
the chain-of-custody log. It then sorts them into two **kits**:

- a **technical** kit, for the question "what did the phone do?"
- a **custody** kit, for the question "how was the evidence handled?"

Both kits include the tool that lists the files. Giving each agent only the kit it needs keeps the
two questions apart. The short sentence under each tool's name is its description: the agent reads
it to decide which tool to use.

**Output:** two lines listing the tool names in each kit. The kits are saved as `technical_tools`
and `custody_tools`; Step A.2 hands them to the agents.

In [ ]:
# Define the evidence tools the agents can use, and sort them into two kits.
import csv   # csv: reads spreadsheet-style files (rows of values separated by commas)
import json  # json: reads and writes structured data (lists and labelled fields)
from pathlib import Path
from IPython.display import Markdown, display  # show formatted text (headings, bold) in the notebook

data_dir = Path('data')   # the folder written in Step 0.3

# A helper (not a tool): open one CSV file from data/ and return all of its rows.
# Each row comes back as a dictionary, for example {'timestamp_utc': ..., 'event': ...}.
def read_csv_rows(filename):
    with (data_dir / filename).open(newline='') as h:
        return list(csv.DictReader(h))

# Each block below is one tool. The @tool line turns the function into a tool the agent may call.
# The sentence in triple quotes is the tool's description. The agent reads it to choose a tool,
# so it is written for the AI as well as for you.
@tool
def list_artifact_files():
    """Return the available artifact files for Lab 05."""
    return sorted(p.name for p in data_dir.iterdir() if p.is_file())

@tool
def get_device_state_events():
    """Return the lock and unlock records that bound the unattended interval."""
    return read_csv_rows('device_state.csv')

@tool
def get_patients_file_event():
    """Return the file event for patients_contacts.png."""
    return read_csv_rows('file_events.csv')

@tool
def get_securechat_attachment_event():
    """Return the SecureChat attachment-attempt record for patients_contacts.png."""
    return read_csv_rows('messaging_events.csv')

@tool
def get_securechat_network_event():
    """Return the SecureChat network upload record."""
    return read_csv_rows('network_events.csv')

@tool
def get_chain_of_custody_events():
    """Return the chain-of-custody records used to judge evidence-handling completeness."""
    return read_csv_rows('chain_of_custody.csv')

# Sort the tools into two kits (lists). The technical kit answers "what did the phone do?".
# The custody kit answers "how was the evidence handled?". Both kits can list the files.
technical_tools = [list_artifact_files, get_device_state_events, get_patients_file_event,
                   get_securechat_attachment_event, get_securechat_network_event]
custody_tools = [list_artifact_files, get_chain_of_custody_events]
# Show which tools are in each kit.
print('technical tools:', [t.name for t in technical_tools])
print('custody tools:  ', [t.name for t in custody_tools])

### Step A.2: Build the three agents and their dependencies

**Purpose:** create the three team members and connect them, so each agent's answer reaches the
agents that need it.

**Input:** the two tool kits from Step A.1, plus `client` and `MODEL` from Step 0.2.

**How it works:**

1. The cell writes down the case question (`LAB_QUESTION`) and the case facts (`CASE_SCOPE`).
   All three agents see the same question and facts.
2. A helper function, `build_forensic_agents()`, creates the three agents. Each one gets a
   **backstory** (a short description of its role), a **task**, the **headings** its answer must
   use, and its **tool kit**. These words are the agent's **prompt** (the instructions we give
   the AI).
3. The `>>` sign wires the handoffs. `investigation >> verification` means "the verifier depends
   on the investigator and will receive its answer". The custody auditor depends on **both**
   earlier agents.

The team is built inside a function so that Part B can build a fresh copy of exactly the same
team.

**Output:** one line: `Custody depends on: ['InvestigationAgent', 'EvidenceVerificationAgent']`.
The three agents are saved as `investigation_agent`, `evidence_verification_agent`, and
`custody_audit_agent`. Steps A.3 to A.5 run them.

In [ ]:
# Build the three agents and connect them into a team.

# The case question all three agents must answer.
LAB_QUESTION = (
    'Was patients_contacts.png transmitted from the outreach phone, should the conclusion be labeled '
    'confirmed, likely, or unconfirmed, and does the missing transfer in the chain-of-custody log '
    'weaken confidence in that conclusion?')

# The case facts all three agents see, including the warning that the custody gap is NOT proof
# about the phone. Everything between the triple quotes is text that is sent to the AI.
CASE_SCOPE = '''Case facts:
- Incident window: 2026-03-01T18:35:00Z to 2026-03-01T19:10:00Z.
- File of interest: patients_contacts.png.
- Technical artifacts: device-state, file, messaging, and network records.
- Evidence-handling review uses chain_of_custody.csv.
- The missing transfer is an undocumented custody handoff between analysts, NOT proof about whether the phone completed transmission.
- The final conclusion must use one label: confirmed, likely, or unconfirmed.'''

# A function (a named, reusable set of steps) that creates the whole team.
# Each Agent gets: a name, a backstory (its role), a task, the headings its answer must use,
# its tool kit, and the AI model and connection from Step 0.2.
def build_forensic_agents():
    # Team member 1: rebuilds the timeline from the phone's records (technical kit).
    investigation = Agent(
        name='InvestigationAgent',
        backstory='You reconstruct mobile-forensics timelines. Order events carefully and do not overclaim upload completion.',
        task_description=f'{LAB_QUESTION}\n\n{CASE_SCOPE}\n\nUse the technical evidence tools to reconstruct the timeline and give an initial transmission assessment.',
        task_expected_output='Use these labels:\nInvestigation timeline:\nInitial transmission assessment:\nOpen technical questions:',
        tools=technical_tools, llm=MODEL, client=client)
    # Team member 2: tests the first agent's claims against the records (technical kit).
    verification = Agent(
        name='EvidenceVerificationAgent',
        backstory='You verify whether technical claims are supported by specific artifact records, and keep ratings evidence-bounded.',
        task_description=f'{LAB_QUESTION}\n\n{CASE_SCOPE}\n\nUse the technical tools to test the earlier claims. If the artifacts show only file creation, an attach attempt, and upload_started, say so. Do not treat upload_started as confirmed completion.',
        task_expected_output='Use these labels:\nClaims checked:\nSupported evidence:\nUnsupported or missing evidence:\nRevised transmission rating:',
        tools=technical_tools, llm=MODEL, client=client)
    # Team member 3: checks how the evidence was handled and writes the final answer (custody kit).
    custody = Agent(
        name='CustodyAuditAgent',
        backstory='You review chain-of-custody records and explain how handling gaps affect final confidence, using earlier technical findings as context.',
        task_description=f'{LAB_QUESTION}\n\n{CASE_SCOPE}\n\nUse the chain-of-custody tool and the earlier context to write the final conclusion. Check for a missing handoff between handlers. Treat any custody gap as an evidence-handling issue, not proof about the phone. Explain how it affects the final label.',
        task_expected_output='Use these labels:\nVerified technical finding:\nCustody review:\nConfidence impact:\nFinal conclusion:',
        tools=custody_tools, llm=MODEL, client=client)
    # Wire the handoffs. "a >> b" means: b depends on a, and b will receive a's answer.
    investigation >> verification
    investigation >> custody
    verification >> custody
    return investigation, verification, custody

# Build the team, then show which agents the custody auditor depends on.
investigation_agent, evidence_verification_agent, custody_audit_agent = build_forensic_agents()
print('Custody depends on:', [a.name for a in custody_audit_agent.dependencies])

### Step A.3: Run InvestigationAgent

**Purpose:** run the first team member. It reads the phone's records and builds a timeline of what
happened while the phone was unattended.

**Input:** `investigation_agent` from Step A.2. It is first in line, so it has no earlier answers
to read; it works only from the evidence tools.

**How it works:** `run()` starts the agent's loop: think, use a tool, read the result, repeat. As it
works, the notebook prints lines such as `Thought:`, `Using Tool:`, and `Tool result:`, so you can
see every record it opens. When it has enough, it writes its answer under the headings it was
given.

**Output:** the agent's working steps, then a formatted answer with three headings:
*Investigation timeline*, *Initial transmission assessment*, and *Open technical questions*.
What to look for: it should call the upload an **attempt** (`upload_started`), not a confirmed
transfer. The answer is saved as `investigation_output`, and it is passed automatically to both
later agents (Steps A.4 and A.5).

In [ ]:
# Run the first agent (the investigator) and show its answer.
# run() lets the agent work through the records. Its answer is also passed on to its teammates.
investigation_output = investigation_agent.run()
# Show the answer as formatted text, under a heading.
display(Markdown('### InvestigationAgent Output\n\n' + investigation_output))

### Step A.4: Run EvidenceVerificationAgent

**Purpose:** run the second team member, the checker. It tests the investigator's claims against the
actual records.

**Input:** `evidence_verification_agent` from Step A.2, plus the investigator's answer from
Step A.3, which it received automatically as context.

**How it works:** the cell first shows you the context the verifier received, so you can see the
handoff with your own eyes. Then the verifier runs its own loop with the technical kit, opening the
records itself, and writes its answer.

**Output:**

1. a box showing the context it received (the investigator's answer),
2. its working steps,
3. its answer under *Claims checked*, *Supported evidence*, *Unsupported or missing evidence*, and
   *Revised transmission rating*.

What to look for: it should lower any claim that treats `upload_started` as a finished transfer.
The answer is saved as `evidence_verification_output` and passed on to the custody auditor in
Step A.5.

In [ ]:
# Show the context the verifier received from the investigator, then run the verifier.
display(Markdown('### EvidenceVerificationAgent received this context\n\n```text\n' + evidence_verification_agent.context.strip() + '\n```'))
# Run the second agent. It checks the earlier claims against the records.
evidence_verification_output = evidence_verification_agent.run()
display(Markdown('### EvidenceVerificationAgent Output\n\n' + evidence_verification_output))

### Step A.5: Run CustodyAuditAgent

**Purpose:** run the third team member. It checks the chain-of-custody log and writes the team's
final conclusion.

**Input:** `custody_audit_agent` from Step A.2, plus **both** earlier answers (from Steps A.3 and
A.4), which it received automatically as context.

**How it works:** the cell shows the context (now two earlier answers), then runs the auditor. It
uses the custody kit, so it reads the chain-of-custody log rather than the phone's records, and it
combines what it finds there with what its teammates found.

**Output:** the context box, the working steps, and the final conclusion under *Verified technical
finding*, *Custody review*, *Confidence impact*, and *Final conclusion*. What to look for: it should
keep the custody gap (Analyst_C began the review with no documented handoff from Analyst_B)
separate from the question of whether the phone sent the file. The answer is saved as
`custody_audit_output`. You will compare it with Part B, and weigh it in Your Final Judgment.

In [ ]:
# Show the context the auditor received (both earlier answers), then run the auditor.
display(Markdown('### CustodyAuditAgent received this context\n\n```text\n' + custody_audit_agent.context.strip() + '\n```'))
# Run the third agent. It reviews the custody log and writes the final conclusion.
custody_audit_output = custody_audit_agent.run()
display(Markdown('### CustodyAuditAgent Final Conclusion\n\n' + custody_audit_output))

## Part B: Orchestrate the Same Team with `Crew`

In Part A you ran each agent yourself and had to keep them in the right order. A **Crew** (a team
manager) does that for you: it knows who depends on whom, works out the order, and runs everyone.
The case, the agents, and their instructions are exactly the same. Only the person pressing "run"
changes.

### Step B.1: Build the team inside a Crew and show the graph

**Purpose:** build a fresh copy of the team inside a Crew, and check the order it plans to run the
agents in.

**Input:** the `build_forensic_agents()` function from Step A.2.

**How it works:** `with Crew() as crew:` opens a Crew, and any agent created inside the indented
block joins it automatically. Calling `build_forensic_agents()` there creates three brand-new
agents, which start with no context (none of Part A's answers). `crew.describe()` then prints who
depends on whom, and the run order the Crew worked out: an agent runs only after everyone it
depends on has finished.

**Output:** the list of dependencies and a line reading
`Run order: InvestigationAgent -> EvidenceVerificationAgent -> CustodyAuditAgent`. The team
manager is saved as `crew`; Step B.2 runs it.

In [ ]:
# Build a fresh team inside a Crew (a team manager), then print its plan.
# Every agent created inside the indented "with" block joins this crew automatically.
with Crew() as crew:
    build_forensic_agents()

# Show who depends on whom, and the order the crew will run the agents in.
print(crew.describe())

### Step B.2: Run the full crew

**Purpose:** let the Crew run the whole team from start to finish in one go.

**Input:** `crew` from Step B.1.

**How it works:** the Crew runs each agent in the planned order, printing a banner
`RUNNING AGENT: <name>` before each one. Each finished answer is passed on to the agents that
depend on it, exactly as you saw in Part A. The Crew collects all three answers.

**Output:** each agent's working steps and printed answer, then the final conclusion (from
CustodyAuditAgent) shown as formatted text. All three answers are saved in `crew_outputs`.
Compare the final conclusion with your Step A.5 result. The wording will differ from run to run
(the AI does not write the same sentences twice). What should not change is that the conclusion
stays within what the evidence shows.

In [ ]:
# Run the whole team in order. crew_outputs keeps each agent's answer, looked up by agent name.
crew_outputs = crew.run()
# Show the final conclusion, which comes from the last agent, CustodyAuditAgent.
display(Markdown('### Crew Final Conclusion (CustodyAuditAgent)\n\n' + crew_outputs['CustodyAuditAgent']))

## Your Final Judgment

The agents are aids, not the authority. You, the examiner, make the call.

**Purpose:** record your own conclusion on the case, in your own words.

**Input:** the case records (see The Case in One Minute, and the tool results printed in Part A),
and the agents' answers from Steps A.3 to A.5 and Step B.2.

**How it works:** in the cell below, replace each `...` text with your own answer (keep the quote
marks around it), then run the cell. It prints your answers back, so they are saved in the
notebook.

**Output:** your four answers, one per line. This is part of what you submit.

Answer these four points:

1. **Transmission label:** confirmed / likely / unconfirmed, and why.
2. **Technical basis:** which records support it (and the `upload_started` vs `upload_completed` point).
3. **Custody impact:** how the missing handoff affects confidence, kept separate from the phone question.
4. **Remaining uncertainty:** what the evidence does not establish.

In [ ]:
# Your final judgment. Replace each '...' text with your own words (keep the quote marks), then run.
final = {
    'transmission_label': '... confirmed / likely / unsupported ...',
    'technical_basis': '... the timestamps and records behind your label ...',
    'custody_impact': '... what the chain-of-custody gap does and does not affect ...',
    'uncertainty': '... what no record confirms ...',
}
# Print each answer so it is saved with the notebook.
for k, v in final.items():
    print(k, '->', v)

### What to notice

1. Did `InvestigationAgent` keep the timeline clear without claiming the upload finished?
2. Did `EvidenceVerificationAgent` lower its confidence when it saw only `upload_started`?
3. Did `CustodyAuditAgent` explain the custody gap without turning it into proof about the phone?
4. Did you keep the transmission question separate from the evidence-handling question?

## Submit

Save the notebook with Part A, Part B, and your judgment filled in.

- In Colab: **File -> Download -> Download .ipynb**, or **File -> Save a copy in Drive**.

---

*Case data and the lab concept are adapted from*
*[frankwxu/agentic-AI4-forensics](https://github.com/frankwxu/agentic-AI4-forensics). The multiagent
framework is adapted from [The Neural Maze's agentic-patterns-course](https://github.com/neural-maze/agentic-patterns-course) (MIT License).*